# 1.1 Runnables & LCEL

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Explain the `Runnable` interface: what `invoke`, `batch`, and `stream` give you on every component.
- Compose runnables with the `|` pipe operator (LCEL) and run branches in parallel with `RunnableParallel`.
- Add retries and fallbacks to a chain with `.with_retry()` and `.with_fallbacks()`.
- Write custom steps with `RunnableLambda` and pass data through with `RunnablePassthrough.assign()`.
- Explain where legacy chain classes went and why new code does not use them.

## The idea in plain words

Everything in LangChain is a `Runnable`: prompts, models, parsers, retrievers, tools, and whole chains. A Runnable is just a contract that says "I accept some input, I produce some output, and I support the same calling conventions as everything else". That contract is what makes composition possible. The pipe operator, `|`, is the syntax for saying "the output of the left becomes the input of the right". LCEL (LangChain Expression Language) is the name for this composition style.

Why does this matter for agents? An agent loop is built out of runnables: a prompt runnable, a model runnable, tool runnables, a parser runnable. Understanding the interface now means Modules 2 and 4 reuse the same mental model instead of teaching you a new one. And practically: once something is a Runnable, batching, streaming, retries, and tracing come free, the same way on every component.

## The Runnable interface

Every Runnable supports three calling conventions, in sync and async form:

In [ ]:
chain.invoke({"text": "Good morning"})    # one input -> one output, blocking
chain.batch([{"text": "Good morning"}, {"text": "Good evening"}])  # many inputs, concurrently
for chunk in chain.stream({"text": "Good morning"}):  # output as it arrives
    print(chunk, end="", flush=True)

The async twins are `ainvoke`, `abatch`, and `astream`. There are also `transform`/`atransform` (stream in, stream out) and `astream_events` (a structured event log of everything the chain did, invaluable for debugging; use `version="v2"`, the current schema).

The point is uniformity. A prompt has `.invoke()`. A model has `.invoke()`. A parser has `.invoke()`. A chain built from them has `.invoke()` with the same shape. You learn the interface once and it covers the whole framework.

## The pipe operator

Piping a prompt into a model into a parser is the canonical first chain:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Translate to French: {text}")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

print(chain.invoke({"text": "Good morning"}))
# Bonjour

Under the hood, `|` calls `Runnable.__or__` and builds a `RunnableSequence`. Each step's output becomes the next step's input, and LangChain coerces between the shapes automatically: the prompt produces a message list, the model consumes messages and produces an `AIMessage`, the parser consumes the message and produces a string.

If you have ever seen `from langchain.chat_models import init_chat_model` in recent code, that is the provider-agnostic alternative: `init_chat_model("openai:gpt-4o-mini")` picks the right integration class for you. Either form is fine; the course uses explicit classes so you always know which provider you are talking to.

## Parallel branches with RunnableParallel

Sometimes steps do not depend on each other. Classify a ticket *and* summarise it, then combine the results. Write it as a dict inside a sequence and LangChain coerces the dict to a `RunnableParallel`, running each branch concurrently on the same input:

In [ ]:
from langchain_core.runnables import RunnableParallel

classifier = ChatPromptTemplate.from_template("Classify this ticket: {ticket}") | llm | StrOutputParser()
summariser = ChatPromptTemplate.from_template("Summarise in one sentence: {ticket}") | llm | StrOutputParser()

combo = RunnableParallel(
    category=classifier,
    summary=summariser,
)
print(combo.invoke({"ticket": "I was charged twice for my subscription"}))
# {'category': '...', 'summary': '...'}

You can also write the dict literal directly in the pipe: `prompt | {"category": classifier, "summary": summariser}`. Each branch runs concurrently; if one branch fails, the parallel step raises by default. Cap concurrency with config: `combo.invoke(input, config={"max_concurrency": 5})`.

Two companions you will reach for constantly:

- `RunnablePassthrough()` passes its input through unchanged. Its classmethod `RunnablePassthrough.assign(summary=summariser)` *adds* computed keys to a dict input without replacing the originals, which is how you enrich data mid-chain: `{"ticket": ..., "summary": ...}` flows onward with both keys.
- `RunnableLambda` wraps any plain function so it can sit in a chain: `RunnableLambda(lambda x: x.strip())`, or as a decorator `@RunnableLambda` on a named function. If the function returns another Runnable, that runnable gets invoked, which is a neat trick for dynamic routing.

A naming trap to save you an hour someday: `RunnableMap` is not a separate class. It is an alias: `RunnableMap = RunnableParallel`. Old tutorials treat them as different things; they are the same thing.

## Retries, fallbacks, and config

Real chains hit rate limits and flaky networks. Two modifiers handle that declaratively:

In [ ]:
resilient_model = ChatOpenAI(model="gpt-4o-mini", max_retries=3)
chain = prompt | resilient_model.with_retry(stop_after_attempt=3) | StrOutputParser()

`.with_retry()` wraps any runnable with tenacity-based retries (exponential backoff with jitter by default). The scope advice from the docs is worth internalising: **retry the smallest thing that can fail**. `prompt | model.with_retry()` retries the model call; putting `.with_retry()` on the whole chain retries the prompt formatting too, which is pointless and can double your input tokens.

`.with_fallbacks()` tries the primary runnable, then each fallback in order:

In [ ]:
primary = ChatOpenAI(model="gpt-4o")
backup = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | primary.with_fallbacks([backup]) | StrOutputParser()

If the primary 500s, the backup runs the same input. With the `exception_key` argument, the caught exception is passed into the fallbacks as a dict, so a fallback can adapt to *why* the primary failed.

`.with_config()` binds run metadata: tags, metadata, `run_name`, timeouts. You will use this mostly for observability in Module 6: tag runs with the module name so LangSmith traces stay searchable.

`.bind()` fixes invocation kwargs: `model.bind(stop=["\n\n"])` returns a runnable that always passes that stop sequence. Small, but it keeps prompt templates clean of provider-specific knobs.

## Seeing inside a chain

Long chains fail opaquely. Three tools, in order of how often you will use them:

1. **`chain.get_graph().print_ascii()`** draws the chain as ASCII. Run it the moment a chain's data flow confuses you; it shows every step and edge, including the implicit `RunnableParallel` nodes that dict literals create.
2. **`chain.astream_events(version="v2")`** yields a structured event for every step start, stream, and end. When output is wrong, the event log shows which step produced the wrong intermediate value. (One caveat as of writing: a bug report against langchain-core 1.6.10 noted v2 emits no terminal error event when a chain step fails, so pair it with try/except rather than relying on events alone for error detection.)
3. **LangSmith tracing** (Module 6) records every run with inputs, outputs, and latency. For anything beyond a toy, this replaces print-debugging.

A common surprise: `stream()` yields provider-shaped chunks (`AIMessageChunk` objects), not strings. Reassemble text with the `+` operator on chunks, or let an output parser do it: `StrOutputParser` supports chunk concatenation, which is why the canonical chain ends with one.

## Where the old chains went

If you follow a tutorial from before late 2025, you will meet `LLMChain`, `SimpleSequentialChain`, `SequentialChain`, `AgentExecutor`, and the old memory classes. In LangChain 1.0 (October 2025) these moved to the **`langchain-classic`** package, which is in maintenance mode. They still work in old projects, but new code uses LCEL composition plus `create_agent` (Module 2) or LangGraph (Module 4). When a tutorial's imports fail on your pinned versions, check whether it is teaching `langchain-classic` APIs; if it is, find a newer tutorial rather than downgrading.

## What can go wrong

- **Opaque intermediate steps.** A five-step chain returns garbage and you cannot tell which step broke. Reach for `get_graph().print_ascii()` and `astream_events` before adding print statements.
- **Retrying the whole chain.** `.with_retry()` on a long chain re-runs cheap steps and can double token spend. Retry the model call, the smallest failing scope.
- **RunnableLambda pickling.** Lambdas and closures do not pickle, which breaks multiprocessing batch workers and some deployment setups. Use named module-level functions wrapped in `RunnableLambda`, not inline lambdas, in anything you intend to deploy.
- **Async event-loop traps.** Calling `ainvoke` from sync code, or `asyncio.run` inside a notebook or a running server, produces the classic "event loop is already running" errors. In scripts, `asyncio.run()` around one top-level coroutine is the clean pattern; in servers, stay async end to end.
- **Streaming chunk confusion.** `stream()` yields `AIMessageChunk`, and naive string concatenation of raw chunks can split mid-token. End chains with a parser that handles chunk assembly, and only drop to raw chunks when you need per-token control.
- **Version confusion from stale tutorials.** Pre-1.0 tutorials teach `langchain-classic` APIs with current-looking confidence. Pin your versions (lesson 0.2) and treat any tutorial that imports `LLMChain` as historical.

## Check your understanding

**1. What does the `|` operator actually build, and what happens to a dict literal inside a sequence?**
`|` invokes `Runnable.__or__` and builds a `RunnableSequence`, where each step's output feeds the next step's input. A dict literal inside a sequence is coerced to a `RunnableParallel`: each value runs concurrently on the same input and the results come back as a dict.

**2. Why does retrying `model.with_retry()` beat putting `.with_retry()` on the whole chain?**
Retry the smallest scope that can fail. Retrying the model call re-sends only that call; retrying the whole chain re-runs prompt formatting and any preceding steps, wasting tokens and potentially repeating side effects.

**3. `RunnableMap` and `RunnableParallel`: same or different?**
The same. `RunnableMap` is an alias (`RunnableMap = RunnableParallel`), a leftover name. Tutorials that treat them as distinct are outdated.

**4. Your chain's final output is wrong. Name the two fastest ways to find which step broke.**
`chain.get_graph().print_ascii()` to see the actual step graph, and `chain.astream_events(version="v2")` to inspect each step's intermediate input/output. LangSmith tracing is the third, for persistent runs.

**5. A pre-2026 tutorial's code fails on your pinned versions with an import error for `LLMChain`. What happened, and what do you do?**
In LangChain 1.0 the legacy chain classes moved to the `langchain-classic` maintenance package. Do not downgrade; rewrite the example with LCEL (`prompt | model | parser`) or find a current tutorial.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.